# Phase 2 — MetSeg 4-Modality Final Training + v2 Embedding Extraction## DynUNet + DenseNet121 Two-Stage CNN — Cyprus PROTEAS → [WT, TC, ET]**Architecture:** Met-Seg (Sadegheih & Merhof, MICCAI 2024 PRIME)- Pretrained on **402 brain metastasis cases** (BraTS-METS 2023)- Two-stage: DenseNet121 detector → DynUNet segmenter- **4 modalities:** T1, T1ce, T2, FLAIR → **3 sub-regions:** WT, TC, ET**Embedding Extraction (v2):**- ROI Crop to WT bounding box + 8px padding → resize to 64³- Octant spatial pooling (8 sub-regions × C channels)- Mask-weighted pooling (WT/TC/ET weighted feature maps)**Strategy:** One fold per Kaggle session. Relaunch auto-continues next fold.

In [ ]:
# ╔════════════════════════════════════════════════════════════╗# ║  CONFIG — MetSeg 4-Modality Final (v4)                   ║# ╚════════════════════════════════════════════════════════════╝MODE = 'train_all'   # 'quick_test' | 'train_all'CV_TYPE = '3fold'CONFIG = {    'seg_params': {        'spatial_dims': 3, 'in_channels': 4, 'out_channels': 3,        'kernel_size': [[3,3,3],[3,3,3],[3,3,3],[3,3,3],[3,3,3]],        'strides': [[1,1,1],[2,2,2],[2,2,2],[2,2,2],[2,2,2]],        'upsample_kernel_size': [[2,2,2],[2,2,2],[2,2,2],[2,2,2]],        'deep_supervision': True, 'deep_supr_num': 3,        'filters': [32, 64, 128, 256, 320], 'res_block': True, 'trans_bias': True,    },    'det_params': {        'spatial_dims': 3, 'in_channels': 4, 'out_channels': 1, 'dropout_prob': 0.2,    },    'patch_size': [64, 64, 64],    'num_samples': 3,    'pos_neg_ratio': [2, 1],    'lr': 2e-4,    'weight_decay': 1e-5,    'cache_rate': 0.5,    'batch_size': 2,    'num_workers': 0,    'warmup_epochs': 3,    'step_epoch_1': 30,    'step_epoch_2': 45,    'unfreeze_det_epoch': 30,    'det_lr': 1e-5,}if MODE == 'quick_test':    CONFIG['epochs'] = 10; CONFIG['patience'] = 10; CONFIG['val_interval'] = 2else:    CONFIG['epochs'] = 60; CONFIG['patience'] = 25; CONFIG['val_interval'] = 5REGION_NAMES = ['WT', 'TC', 'ET']ROI_SIZE = (64, 64, 64)ROI_PADDING = 8print(f'Mode: {MODE} | CV: {CV_TYPE}')print(f'Epochs: {CONFIG["epochs"]} | LR: {CONFIG["lr"]} | Batch: {CONFIG["batch_size"]}')print(f'Extraction: ROI crop ({ROI_PADDING}px pad) → {ROI_SIZE} → Octant + Mask-weighted pooling')print(f'Strategy: ONE fold per session → relaunch auto-continues')

In [ ]:
# ── Install dependencies ──import subprocess, sysfor pkg in ['monai', 'nibabel', 'pytorch-lightning', 'easydict']:    try: __import__(pkg.replace('-', '_'))    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])print('Dependencies installed ✅')

In [ ]:
import os, shutilfrom pathlib import PathREPO_DIR = Path('/kaggle/working/Met-Seg')if not REPO_DIR.exists():    os.system('git clone https://github.com/xmindflow/Met-Seg.git /kaggle/working/Met-Seg')    print('Cloned Met-Seg repo ✅')else:    print('Met-Seg repo already exists ✅')import syssys.path.insert(0, str(REPO_DIR / 'src'))print(f'Repo at: {REPO_DIR}')

In [ ]:
import warningswarnings.filterwarnings('ignore', message='.*Num foregrounds 0.*')warnings.filterwarnings('ignore', message='.*non-tuple sequence.*')warnings.filterwarnings('ignore', message='.*axcodes.*length.*')warnings.filterwarnings('ignore', message='.*FutureWarning.*')import torchimport torch.nn as nnimport torch.nn.functional as Fimport numpy as npimport json, timeimport matplotlib; matplotlib.use('Agg')import matplotlib.pyplot as pltfrom pathlib import Pathfrom collections import OrderedDictfrom tqdm import tqdmfrom monai.networks.nets import DynUNet, DenseNet121from monai.losses import DiceLossfrom torch.nn import BCEWithLogitsLossfrom monai.data import DataLoader, CacheDatasetfrom monai.inferers import sliding_window_inferencefrom monai.metrics import DiceMetricimport monai.transforms as Tfrom monai.transforms.compose import MapTransformfrom monai.utils import ensure_tuple_repdevice = torch.device('cuda' if torch.cuda.is_available() else 'cpu')print(f'Device: {device}')if torch.cuda.is_available():    print(f'GPU: {torch.cuda.get_device_name(0)}')    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

In [ ]:
# ── Download / locate pretrained weights ──OUTPUT_ROOT = Path('/kaggle/working/phase2_metseg_outputs')WEIGHTS_DIR = OUTPUT_ROOT / 'pretrained_weights'WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)WEIGHT_FILES = {'segmentor': 'segmentor_full_modality.ckpt', 'detector': 'detector_full_modality.ckpt'}WEIGHT_URLS = {    'segmentor': 'https://myfiles.uni-regensburg.de/filr/public-link/file-download/0447879c90b809a80190bbf452df0f6b/120822/-2821463328806488135/segmentor_weight%20%28full%20modality%29.ckpt',    'detector': 'https://myfiles.uni-regensburg.de/filr/public-link/file-download/0447879c90b809a80190bbf45b1d0f6f/120821/-4880761362689834300/detector_weight%20%28full%20modality%29.ckpt',}KAGGLE_WEIGHT_DIRS = [    Path('/kaggle/input/datasets/zinou123viva/metseg-pretrained-weights'),    Path('/kaggle/input/metseg-weights'), Path('/kaggle/input/met-seg-weights'),]def find_weight_in_kaggle(name):    for d in KAGGLE_WEIGHT_DIRS:        if not d.exists(): continue        for f in d.rglob('*.ckpt'):            if name in f.name.lower(): return f    return Nonefor name, filename in WEIGHT_FILES.items():    dst = WEIGHTS_DIR / filename    if dst.exists() and dst.stat().st_size > 1_000_000:        print(f'  ✅ {name}: {dst.stat().st_size/1024/1024:.1f} MB (cached)'); continue    kaggle_path = find_weight_in_kaggle(name)    if kaggle_path:        shutil.copy2(kaggle_path, dst)        print(f'  ✅ {name}: {dst.stat().st_size/1024/1024:.1f} MB (from Kaggle)'); continue    print(f'  Downloading {name} via wget...')    result = subprocess.run(['wget', '-q', '--no-check-certificate', '--user-agent',        'Mozilla/5.0 (X11; Linux x86_64)', '-O', str(dst), WEIGHT_URLS[name]],        capture_output=True, text=True, timeout=300)    if result.returncode == 0 and dst.exists() and dst.stat().st_size > 1_000_000:        print(f'  ✅ {name}: {dst.stat().st_size/1024/1024:.1f} MB (downloaded)'); continue    print(f'  ❌ Could not get {name} weights! Upload as Kaggle dataset.')print(f'\nWeights dir: {WEIGHTS_DIR}')for f in WEIGHTS_DIR.glob('*.ckpt'): print(f'  {f.name}: {f.stat().st_size/1024/1024:.1f} MB')

In [ ]:
# ── Data path resolution + splits loading ──DATA_ROOT = Path('/kaggle/input/datasets/zinou123viva/cyprus-proteas-brain-mets')if not DATA_ROOT.exists():    for candidate in Path('/kaggle/input').iterdir():        if not candidate.is_dir(): continue        if any((candidate / f'P{i:02d}').exists() for i in range(1, 5)):            DATA_ROOT = candidate; break        if (candidate / 'data_splits.json').exists():            DATA_ROOT = candidate; break        for sub in candidate.iterdir():            if sub.is_dir() and (sub / 'data_splits.json').exists():                DATA_ROOT = sub; breakSYMLINK_DIR = Path('/kaggle/working/nifti_links')def resolve_path(root, rel):    p = root / rel    if p.exists(): return str(p)    gz = str(p) + '.gz'    if Path(gz).exists(): return gz    nii_gz = str(p).replace('.nii.gz', '.nii_gz')    if Path(nii_gz).exists():        link = SYMLINK_DIR / rel; link.parent.mkdir(parents=True, exist_ok=True)        if not link.exists(): os.symlink(nii_gz, str(link))        return str(link)    parent = p.parent    if parent.exists():        for f in parent.iterdir():            if f.name.lower() == p.name.lower(): return str(f)    raise FileNotFoundError(f'Not found: {rel}')# ── Load splits (prefer 4-variable balanced) ──def find_best_splits():    candidates = []    for name in ['data_splits.json', 'cv_splits_3fold.json']:        p = DATA_ROOT / name        if p.exists(): candidates.append(p)    for f in DATA_ROOT.rglob('*splits*.json'):        if f not in candidates: candidates.append(f)    for kaggle_dir in Path('/kaggle/input').iterdir():        if not kaggle_dir.is_dir(): continue        p = kaggle_dir / 'data_splits.json'        if p.exists() and p not in candidates: candidates.append(p)        for sub in kaggle_dir.iterdir():            if sub.is_dir():                p = sub / 'data_splits.json'                if p.exists() and p not in candidates: candidates.append(p)    for c in candidates:        try:            d = json.load(open(c))            if len(d.get('metadata', {}).get('stratification_variables', [])) >= 4:                return c, d, 'NEW (4-variable balanced)'        except: pass    for c in candidates:        try:            d = json.load(open(c))            if '3fold' in d or 'fold_0' in d: return c, d, 'OLD (2-variable)'        except: pass    raise FileNotFoundError('No valid splits found')splits_path, raw_splits, splits_type = find_best_splits()all_splits = raw_splits.get('3fold', raw_splits)if 'fold_0' not in all_splits:    all_splits = {k: v for k, v in raw_splits.items() if k.startswith('fold_')}meta = raw_splits.get('metadata', {})print(f'DATA_ROOT: {DATA_ROOT}')print(f'Splits: {splits_path.name} ({splits_type})')print(f'Folds: {list(all_splits.keys())}')

In [ ]:
# ── Label conversion: Cyprus {0,1,2,3} → BraTS [WT, TC, ET] ──class ConvertToMultiChannelBratsMetsd(MapTransform):    def __call__(self, data):        d = dict(data)        for key in self.key_iterator(d):            img = d[key]            if img.ndim == 4 and img.shape[0] == 1: img = img.squeeze(0)            result = [                (img == 1) | (img == 3) | (img == 2),  # WT = all tumor                (img == 1) | (img == 3),                 # TC = core                img == 3,                                 # ET = enhancing            ]            d[key] = (torch.stack(result, dim=0).float() if isinstance(img, torch.Tensor)                      else np.stack(result, axis=0).astype(np.float32))        return dpatch = CONFIG['patch_size']train_transforms = T.Compose([    T.LoadImaged(keys=['image', 'label']),    T.EnsureChannelFirstd(keys=['image', 'label']),    T.EnsureTyped(keys=['image', 'label']),    T.Orientationd(keys=['image', 'label'], axcodes='RAS'),    T.CropForegroundd(keys=['image', 'label'], source_key='image', allow_smaller=True),    T.NormalizeIntensityd(keys='image', nonzero=True, channel_wise=True),    ConvertToMultiChannelBratsMetsd(keys=['label']),    T.RandFlipd(keys=['image', 'label'], spatial_axis=[0], prob=0.5),    T.RandFlipd(keys=['image', 'label'], spatial_axis=[1], prob=0.5),    T.RandFlipd(keys=['image', 'label'], spatial_axis=[2], prob=0.5),    T.RandScaleIntensityd(keys='image', factors=0.1, prob=0.3),    T.SpatialPadd(keys=['image', 'label'], spatial_size=ensure_tuple_rep(patch, 3)),    T.RandCropByPosNegLabeld(keys=['image', 'label'], label_key='label',        spatial_size=ensure_tuple_rep(patch, 3),        pos=CONFIG['pos_neg_ratio'][0], neg=CONFIG['pos_neg_ratio'][1],        num_samples=CONFIG['num_samples'], image_key='image', image_threshold=0),    T.EnsureTyped(keys=['image', 'label'], dtype=torch.float32),])val_transforms = T.Compose([    T.LoadImaged(keys=['image', 'label']),    T.EnsureChannelFirstd(keys=['image', 'label']),    T.EnsureTyped(keys=['image', 'label']),    T.Orientationd(keys=['image', 'label'], axcodes='RAS'),    T.CropForegroundd(keys=['image', 'label'], source_key='image', allow_smaller=True),    T.NormalizeIntensityd(keys='image', nonzero=True, channel_wise=True),    ConvertToMultiChannelBratsMetsd(keys=['label']),    T.EnsureTyped(keys=['image', 'label'], dtype=torch.float32),])print('Label mapping: ch0=WT (1+2+3) | ch1=TC (1+3) | ch2=ET (3)')print('Transforms defined ✅')

In [ ]:
def build_scan_dicts(scans, label='subset'):    dicts, skips = [], 0    for scan in scans:        try:            dicts.append({                'image': [resolve_path(DATA_ROOT, scan['t1']), resolve_path(DATA_ROOT, scan['t1c']),                          resolve_path(DATA_ROOT, scan['t2']), resolve_path(DATA_ROOT, scan['fla'])],                'label': resolve_path(DATA_ROOT, scan['mask']),                'patient_dir': scan['patient_dir'], 'visit': scan['visit'],            })        except FileNotFoundError: skips += 1    if skips: print(f'  Skipped {skips} in {label}')    return dictsdef get_fold_dicts(fold):    fd = all_splits[f'fold_{fold}']    return build_scan_dicts(fd['train_scans'], f'fold{fold}_train'), build_scan_dicts(fd['test_scans'], f'fold{fold}_val')def get_all_dicts():    all_d, seen = [], set()    for fk in all_splits:        for scan in all_splits[fk]['train_scans'] + all_splits[fk]['test_scans']:            key = (scan['patient_dir'], scan['visit'])            if key not in seen:                seen.add(key)                try:                    all_d.append({'image': [resolve_path(DATA_ROOT, scan['t1']), resolve_path(DATA_ROOT, scan['t1c']),                                            resolve_path(DATA_ROOT, scan['t2']), resolve_path(DATA_ROOT, scan['fla'])],                                  'label': resolve_path(DATA_ROOT, scan['mask']),                                  'patient_dir': scan['patient_dir'], 'visit': scan['visit']})                except FileNotFoundError: pass    return all_dtrain_dicts, val_dicts = get_fold_dicts(0)print(f'Fold 0: {len(train_dicts)} train | {len(val_dicts)} val')print(f'Total scans (all folds): {len(get_all_dicts())}')

In [ ]:
# ── Model creation & weight loading ──def create_segmenter(): return DynUNet(**CONFIG['seg_params'])def create_detector(): return DenseNet121(**CONFIG['det_params'])def load_lightning_weights(model, ckpt_path, prefix='model.'):    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)    sd = ckpt.get('state_dict', ckpt)    new_sd = OrderedDict()    for k, v in sd.items():        new_key = k[len(prefix):] if k.startswith(prefix) else k        new_sd[new_key] = v    missing, unexpected = model.load_state_dict(new_sd, strict=False)    return missing, unexpecteddef find_weight(name):    candidate = WEIGHTS_DIR / f'{name}_full_modality.ckpt'    if candidate.exists() and candidate.stat().st_size > 1_000_000: return candidate    import glob    for pattern in [f'/kaggle/input/**/{name}*.ckpt', f'/kaggle/input/**/*{name}*.ckpt']:        for m in glob.glob(pattern, recursive=True):            if os.path.getsize(m) > 1_000_000: return Path(m)    return Noneseg_weights = find_weight('segmentor')det_weights = find_weight('detector')seg_test = create_segmenter()det_test = create_detector()if seg_weights:    sm, su = load_lightning_weights(seg_test, seg_weights)    print(f'Segmenter: Missing={len(sm)} | Unexpected={len(su)}')if det_weights:    dm, du = load_lightning_weights(det_test, det_weights)    print(f'Detector: Missing={len(dm)} | Unexpected={len(du)}')sp = sum(p.numel() for p in seg_test.parameters())dp = sum(p.numel() for p in det_test.parameters())print(f'Segmenter: {sp:,} params ({sp/1e6:.1f}M) | Detector: {dp:,} params ({dp/1e6:.1f}M)')del seg_test, det_testprint('✅ Pretrained weights verified')

In [ ]:
def get_lr_for_epoch(epoch, config):    warmup = config['warmup_epochs']    base_lr = config['lr']    if epoch < warmup: return 1e-5 + (base_lr - 1e-5) * (epoch / warmup)    elif epoch < config['step_epoch_1']: return base_lr    elif epoch < config['step_epoch_2']: return 5e-5    else: return 1e-5def train_fold(fold):    ckpt_dir = OUTPUT_ROOT / 'checkpoints'; ckpt_dir.mkdir(parents=True, exist_ok=True)    best_path = ckpt_dir / f'metseg_fold{fold}_best.pth'    latest_path = ckpt_dir / f'metseg_fold{fold}_latest.pth'    # Check if already complete    if best_path.exists() and latest_path.exists():        lc = torch.load(latest_path, map_location='cpu', weights_only=False)        if lc.get('epoch', -1) >= CONFIG['epochs'] - 1:            bd = lc.get('best_dice', 0)            print(f'  ✅ Fold {fold} already complete (Dice={bd:.4f})')            seg = create_segmenter().to(device)            seg.load_state_dict(torch.load(best_path, map_location=device, weights_only=False)['seg_state_dict'])            return seg, bd, lc.get('metrics_history', {}), True    train_dicts, val_dicts = get_fold_dicts(fold)    print(f'  Train: {len(train_dicts)} | Val: {len(val_dicts)}')    train_ds = CacheDataset(train_dicts, train_transforms, cache_rate=CONFIG['cache_rate'], num_workers=CONFIG['num_workers'])    val_ds = CacheDataset(val_dicts, val_transforms, cache_rate=1.0, num_workers=CONFIG['num_workers'])    train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=CONFIG['num_workers'], pin_memory=True)    val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=CONFIG['num_workers'])    seg_model = create_segmenter().to(device)    det_model = create_detector().to(device)    if seg_weights: load_lightning_weights(seg_model, seg_weights); print('  Segmenter pretrained ✅')    if det_weights: load_lightning_weights(det_model, det_weights); print('  Detector pretrained ✅')    det_model.eval()    for p in det_model.parameters(): p.requires_grad = False    det_unfrozen = False    optimizer = torch.optim.AdamW(seg_model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])    dice_loss_fn = DiceLoss(sigmoid=True, smooth_nr=0, smooth_dr=1e-5)    bce_loss_fn = BCEWithLogitsLoss()    scaler = torch.amp.GradScaler('cuda')    dice_metric = DiceMetric(include_background=True, reduction='mean_batch')    best_dice, patience_ctr, start_epoch = 0.0, 0, 0    metrics_log = {'train_loss': [], 'val_dice': [], 'val_per_region': [], 'lr': []}    # Resume from latest    if latest_path.exists():        ckpt = torch.load(latest_path, map_location=device, weights_only=False)        seg_model.load_state_dict(ckpt['seg_state_dict'])        optimizer.load_state_dict(ckpt['optimizer_state_dict'])        start_epoch = ckpt['epoch'] + 1; best_dice = ckpt.get('best_dice', 0.0)        metrics_log = ckpt.get('metrics_history', metrics_log)        patience_ctr = ckpt.get('patience_ctr', 0); det_unfrozen = ckpt.get('det_unfrozen', False)        if det_unfrozen and 'det_state_dict' in ckpt:            det_model.load_state_dict(ckpt['det_state_dict'])            for p in det_model.parameters(): p.requires_grad = True            optimizer.add_param_group({'params': det_model.parameters(), 'lr': CONFIG['det_lr'], 'weight_decay': CONFIG['weight_decay']})        print(f'  🔄 Resuming from epoch {start_epoch} (best={best_dice:.4f})')    def get_sd(m): return m.module.state_dict() if hasattr(m, 'module') else m.state_dict()    t0 = time.time()    for epoch in range(start_epoch, CONFIG['epochs']):        lr = get_lr_for_epoch(epoch, CONFIG)        for pg in optimizer.param_groups:            if pg.get('lr', 0) > CONFIG.get('det_lr', 1e-5) or not det_unfrozen: pg['lr'] = lr        if epoch >= CONFIG['unfreeze_det_epoch'] and not det_unfrozen:            for p in det_model.parameters(): p.requires_grad = True            optimizer.add_param_group({'params': det_model.parameters(), 'lr': CONFIG['det_lr'], 'weight_decay': CONFIG['weight_decay']})            det_unfrozen = True            print(f'  🔓 Detector unfrozen at epoch {epoch}')        seg_model.train()        if det_unfrozen: det_model.train()        ep_loss, n = 0.0, 0        for batch in train_loader:            imgs, lbls = batch['image'].to(device), batch['label'].to(device)            if det_unfrozen: det_pred = det_model(imgs)            else:                with torch.no_grad(): det_pred = det_model(imgs)            mask = torch.sigmoid(det_pred.squeeze(-1)) > 0.3            if mask.sum() == 0: mask = torch.ones(len(imgs), dtype=torch.bool, device=device)            ci, cl = imgs[mask], lbls[mask]            optimizer.zero_grad()            with torch.amp.autocast('cuda'):                out = seg_model(ci)                if isinstance(out, (list, tuple)):                    loss = sum(0.5**i * (dice_loss_fn(p, cl) + bce_loss_fn(p, cl)) for i, p in enumerate(out))                elif out.dim() == 6:                    preds = torch.unbind(out, dim=1)                    loss = sum(0.5**i * (dice_loss_fn(p, cl) + bce_loss_fn(p, cl)) for i, p in enumerate(preds))                else: loss = dice_loss_fn(out, cl) + bce_loss_fn(out, cl)            scaler.scale(loss).backward()            scaler.unscale_(optimizer)            torch.nn.utils.clip_grad_norm_(seg_model.parameters(), max_norm=12.0)            scaler.step(optimizer); scaler.update()            ep_loss += loss.item(); n += 1        avg_loss = ep_loss / max(n, 1)        metrics_log['train_loss'].append(avg_loss); metrics_log['lr'].append(lr)        if (epoch + 1) % CONFIG['val_interval'] == 0 or epoch == CONFIG['epochs'] - 1:            seg_model.eval(); dice_metric.reset()            with torch.no_grad():                for vb in val_loader:                    vi, vl = vb['image'].to(device), vb['label'].to(device)                    vo = sliding_window_inference(vi, CONFIG['patch_size'], 4, seg_model, overlap=0.5, mode='gaussian')                    if isinstance(vo, (list, tuple)): vo = vo[0]                    if vo.dim() == 6: vo = vo[:, 0]                    dice_metric((torch.sigmoid(vo) > 0.5).float(), vl)            dv = dice_metric.aggregate(); md = dv.mean().item()            pr = [dv[i].item() for i in range(len(REGION_NAMES))]            metrics_log['val_dice'].append(md); metrics_log['val_per_region'].append(pr)            elapsed = (time.time() - t0) / 60            rs = ' '.join([f'{n}={v:.3f}' for n, v in zip(REGION_NAMES, pr)])            print(f'Ep {epoch:3d}/{CONFIG["epochs"]-1} | Loss={avg_loss:.4f} | Dice={md:.4f} ({rs}) | LR={lr:.1e} | {elapsed:.1f}min')            if md > best_dice:                best_dice = md; patience_ctr = 0                torch.save({'epoch': epoch, 'best_dice': best_dice, 'seg_state_dict': get_sd(seg_model),                    'det_state_dict': get_sd(det_model), 'optimizer_state_dict': optimizer.state_dict(),                    'metrics_history': metrics_log, 'det_unfrozen': det_unfrozen, 'config': CONFIG}, best_path)                print(f'  ✅ New best!')            else:                patience_ctr += 1                if patience_ctr >= CONFIG['patience']: print(f'  Early stop ep {epoch}'); break        else:            elapsed = (time.time() - t0) / 60            print(f'Ep {epoch:3d}/{CONFIG["epochs"]-1} | Loss={avg_loss:.4f} | LR={lr:.1e} | {elapsed:.1f}min')        torch.save({'epoch': epoch, 'best_dice': best_dice, 'seg_state_dict': get_sd(seg_model),            'det_state_dict': get_sd(det_model), 'optimizer_state_dict': optimizer.state_dict(),            'metrics_history': metrics_log, 'patience_ctr': patience_ctr,            'det_unfrozen': det_unfrozen, 'config': CONFIG}, latest_path)    elapsed = (time.time() - t0) / 60    print(f'\n  Fold {fold} done: Best Dice = {best_dice:.4f} | {elapsed:.1f} min')    fig_dir = OUTPUT_ROOT / 'figures'; fig_dir.mkdir(parents=True, exist_ok=True)    fig, axes = plt.subplots(1, 3, figsize=(18, 5))    axes[0].plot(metrics_log['train_loss']); axes[0].set_title(f'Loss (Fold {fold})')    if metrics_log['val_dice']: axes[1].plot(metrics_log['val_dice'], marker='o')    axes[1].set_title(f'Val Dice (Fold {fold})')    if metrics_log['lr']: axes[2].plot(metrics_log['lr']); axes[2].set_title('LR Schedule')    plt.tight_layout(); plt.savefig(fig_dir / f'metseg_fold{fold}_curves.png', dpi=150); plt.close()    met_dir = OUTPUT_ROOT / 'metrics'; met_dir.mkdir(parents=True, exist_ok=True)    with open(met_dir / f'metseg_fold{fold}_metrics.json', 'w') as f: json.dump(metrics_log, f, indent=2)    if hasattr(seg_model, 'module'): seg_model = seg_model.module    return seg_model, best_dice, metrics_log, Falseprint('Train function ready ✅')

In [ ]:
# ══════════════════════════════════════════════════════════════#  v2 EMBEDDING EXTRACTION — ROI Crop + Octant + Mask-weighted# ══════════════════════════════════════════════════════════════def roi_crop_and_resize(image, label):    wt = label[0, 0]    nz = wt.nonzero(as_tuple=False)    if len(nz) == 0:        img_crop, lbl_crop = image, label    else:        lo = nz.min(0).values; hi = nz.max(0).values        sh = torch.tensor(wt.shape, device=wt.device)        lo = torch.clamp(lo - ROI_PADDING, min=0)        hi = torch.clamp(hi + ROI_PADDING + 1, max=sh)        img_crop = image[:, :, lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]]        lbl_crop = label[:,  :, lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]]    img_roi = F.interpolate(img_crop, size=ROI_SIZE, mode='trilinear', align_corners=False)    lbl_roi = F.interpolate(lbl_crop.float(), size=ROI_SIZE, mode='nearest')    return img_roi, lbl_roidef octant_pool(feat):    H, W, D = feat.shape[2], feat.shape[3], feat.shape[4]    pieces = []    for hs in [slice(None, H//2), slice(H//2, None)]:        for ws in [slice(None, W//2), slice(W//2, None)]:            for ds in [slice(None, D//2), slice(D//2, None)]:                pieces.append(F.adaptive_avg_pool3d(feat[:, :, hs, ws, ds], 1).flatten())    return torch.cat(pieces)def mask_weighted_pool(feat, lbl_roi):    H, W, D = feat.shape[2], feat.shape[3], feat.shape[4]    regions = []    for ch in range(3):        prob = F.interpolate(lbl_roi[:, ch:ch+1], size=(H, W, D), mode='nearest')        w_sum = (feat * prob).sum(dim=[0, 2, 3, 4])        regions.append(w_sum / (prob.sum() + 1e-6))    return torch.cat(regions)def extract_embeddings_v2(model, fold_label):    model.eval(); model.to(device)    _feats = {}    def hook_oct(m, inp, out):        _feats['oct'] = (out[0] if isinstance(out, (list, tuple)) else out).detach()    def hook_neck(m, inp, out):        _feats['neck'] = (out[0] if isinstance(out, (list, tuple)) else out).detach()    h_oct = model.downsamples[-1].register_forward_hook(hook_oct)    h_neck = model.bottleneck.register_forward_hook(hook_neck) if hasattr(model, 'bottleneck') else None    all_d = get_all_dicts()    dataset = CacheDataset(all_d, val_transforms, cache_rate=0.3, num_workers=0)    loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)    embeddings, skipped = {}, 0    with torch.no_grad():        for i, batch in enumerate(tqdm(loader, desc=f'Extracting fold {fold_label}')):            image, label = batch['image'].to(device), batch['label'].to(device)            key = f"{batch['patient_dir'][0]}__{batch['visit'][0]}"            try:                image_roi, label_roi = roi_crop_and_resize(image, label)                _feats.clear()                _ = model(image_roi)                oct_feat = _feats.get('oct')                neck_feat = _feats.get('neck', oct_feat)                if oct_feat is None: skipped += 1; continue                oct_emb = octant_pool(oct_feat)                mask_emb = mask_weighted_pool(neck_feat, label_roi)                embeddings[key] = torch.cat([oct_emb, mask_emb]).cpu().numpy()                if i < 3: print(f'  {key}: {embeddings[key].shape[0]}-dim (oct={oct_emb.shape[0]} + mask={mask_emb.shape[0]})')            except Exception as e: print(f'  ⚠️ {key}: {e}'); skipped += 1    h_oct.remove()    if h_neck: h_neck.remove()    print(f'  Extracted: {len(embeddings)} | Skipped: {skipped}')    emb_dir = OUTPUT_ROOT / 'embeddings'; emb_dir.mkdir(parents=True, exist_ok=True)    np.savez(emb_dir / f'cnn_metseg_embeddings_fold{fold_label}_v2.npz', **embeddings)    meta_d = {k: {'patient_dir': k.split('__')[0], 'visit': k.split('__')[1],        'embedding_dim': int(list(embeddings.values())[0].shape[0])} for k in embeddings}    with open(emb_dir / f'cnn_metseg_embeddings_fold{fold_label}_v2_meta.json', 'w') as f:        json.dump(meta_d, f, indent=2)    dim = list(embeddings.values())[0].shape[0]    print(f'  ✅ Saved: cnn_metseg_embeddings_fold{fold_label}_v2.npz ({len(embeddings)} × {dim}-dim)')    return embeddingsprint('v2 extraction functions ready ✅')

In [ ]:
# ── 3D Inference Visualization ──def visualize_3d_predictions(model, fold, n_samples=3):    model.eval(); model.to(device)    _, vd = get_fold_dicts(fold)    vis_ds = CacheDataset(vd[:n_samples], val_transforms, cache_rate=1.0, num_workers=0)    vis_loader = DataLoader(vis_ds, batch_size=1, shuffle=False, num_workers=0)    fig_dir = OUTPUT_ROOT / 'figures'; fig_dir.mkdir(parents=True, exist_ok=True)    for i, batch in enumerate(vis_loader):        vi = batch['image'].to(device); vl = batch['label'].to(device)        with torch.no_grad():            vo = sliding_window_inference(vi, CONFIG['patch_size'], 4, model, overlap=0.5, mode='gaussian')            if isinstance(vo, (list, tuple)): vo = vo[0]            if vo.dim() == 6: vo = vo[:, 0]        pred = (torch.sigmoid(vo) > 0.5).float().cpu().numpy()[0]        gt = vl.cpu().numpy()[0]        img = vi.cpu().numpy()[0, 1]  # T1c channel        mid = img.shape[2] // 2        fig, axes = plt.subplots(2, 3, figsize=(15, 10))        for c, rn in enumerate(REGION_NAMES):            axes[0, c].imshow(img[:, :, mid], cmap='gray'); axes[0, c].imshow(gt[c, :, :, mid], alpha=0.3, cmap='Reds')            axes[0, c].set_title(f'GT {rn}'); axes[0, c].axis('off')            axes[1, c].imshow(img[:, :, mid], cmap='gray'); axes[1, c].imshow(pred[c, :, :, mid], alpha=0.3, cmap='Blues')            axes[1, c].set_title(f'Pred {rn}'); axes[1, c].axis('off')        pid = batch['patient_dir'][0]; visit = batch['visit'][0]        fig.suptitle(f'{pid} / {visit}', fontsize=14)        plt.tight_layout(); plt.savefig(fig_dir / f'metseg_fold{fold}_3d_sample{i}.png', dpi=150); plt.close()        print(f'  Saved: metseg_fold{fold}_3d_sample{i}.png')    print(f'  ✅ {n_samples} 3D visualizations saved')print('Visualization function ready ✅')

In [ ]:
# ╔════════════════════════════════════════════════════════════╗# ║  MAIN — FOLD-BY-FOLD (one fold per session, auto-resume) ║# ╚════════════════════════════════════════════════════════════╝import shutil as _shutilckpt_dir = OUTPUT_ROOT / 'checkpoints'; ckpt_dir.mkdir(parents=True, exist_ok=True)emb_dir = OUTPUT_ROOT / 'embeddings'; emb_dir.mkdir(parents=True, exist_ok=True)# ── Recover checkpoints from previous Kaggle sessions ──print('Scanning /kaggle/input for previous fold outputs...')_input_root = Path('/kaggle/input')_recovered = []for _f in sorted(_input_root.rglob('metseg_fold*.pth')):    _dest = ckpt_dir / _f.name    if not _dest.exists():        _shutil.copy2(str(_f), str(_dest)); _recovered.append(_f.name)        print(f'  Copied: {_f.name}')for _pat in ['cnn_metseg_embeddings_fold*.npz', 'cnn_metseg_embeddings_fold*_meta.json',             'cnn_metseg_embeddings_fold*_v2.npz', 'cnn_metseg_embeddings_fold*_v2_meta.json']:    for _f in sorted(_input_root.rglob(_pat)):        _dest = emb_dir / _f.name        if not _dest.exists():            _shutil.copy2(str(_f), str(_dest)); _recovered.append(_f.name)            print(f'  Copied: {_f.name}')if _recovered: print(f'  Recovered: {len(_recovered)} files')else: print('  Nothing to recover (session 1)')# ── Find next fold to train ──completed_folds = {}; target_fold = Nonefor f in [0, 1, 2]:    bp = ckpt_dir / f'metseg_fold{f}_best.pth'    lp = ckpt_dir / f'metseg_fold{f}_latest.pth'    if bp.exists() and lp.exists():        ckpt = torch.load(lp, map_location='cpu', weights_only=False)        if ckpt.get('epoch', -1) >= CONFIG['epochs'] - 1:            completed_folds[f] = ckpt.get('best_dice', 0)            print(f'  ✅ Fold {f}: COMPLETE (Dice={completed_folds[f]:.4f})')            continue        else: print(f'  🔄 Fold {f}: PARTIAL (epoch {ckpt.get("epoch", 0)})')    else: print(f'  🆕 Fold {f}: NOT STARTED')    if target_fold is None: target_fold = fif target_fold is None:    print('\n' + '=' * 60); print('  🎉 ALL 3 FOLDS COMPLETE!'); print('=' * 60)    for f, d in completed_folds.items(): print(f'  Fold {f}: Dice={d:.4f}')    print(f'  Mean Dice: {sum(completed_folds.values()) / len(completed_folds):.4f}')    # Extract v2 embeddings for any missing folds    for f in [0, 1, 2]:        v2_path = emb_dir / f'cnn_metseg_embeddings_fold{f}_v2.npz'        if not v2_path.exists():            print(f'\n  Extracting v2 embeddings for fold {f}...')            seg = create_segmenter().to(device)            seg.load_state_dict(torch.load(ckpt_dir / f'metseg_fold{f}_best.pth', map_location=device, weights_only=False)['seg_state_dict'])            extract_embeddings_v2(seg, f)            del seg; torch.cuda.empty_cache()else:    print(f'\n▶ Training FOLD {target_fold}')    print('=' * 60)    model, best_dice, metrics, was_skipped = train_fold(target_fold)    # Load best weights and extract v2 embeddings    bp = ckpt_dir / f'metseg_fold{target_fold}_best.pth'    if bp.exists():        model.load_state_dict(torch.load(bp, map_location=device, weights_only=False)['seg_state_dict'])    print(f'\n  Extracting v2 embeddings for fold {target_fold}...')    extract_embeddings_v2(model, target_fold)    print(f'\n  Generating 3D visualizations...')    visualize_3d_predictions(model, target_fold)    completed_folds[target_fold] = best_dice    remaining = [f for f in [0,1,2] if f not in completed_folds]    print('\n' + '=' * 60)    print(f'  ✅ Fold {target_fold} DONE — Dice={best_dice:.4f}')    print(f'  Completed: {list(completed_folds.keys())} | Remaining: {remaining}')    if remaining: print(f'  → Relaunch to train fold {remaining[0]}')    else:        print(f'  🎉 ALL FOLDS COMPLETE!')        print(f'  Mean Dice: {sum(completed_folds.values()) / len(completed_folds):.4f}')    print('=' * 60)# ── Output summary ──print(f'\n  Files saved to {OUTPUT_ROOT}:')total = 0for p in sorted(OUTPUT_ROOT.rglob('*')):    if p.is_file():        sz = p.stat().st_size; total += sz        print(f'    {p.relative_to(OUTPUT_ROOT)} ({sz/1024/1024:.1f} MB)')print(f'\n  Total: {total/1024/1024:.1f} MB / 19,000 MB limit')

In [ ]:
# ── Embedding Verification & Diversity Check ──import randomprint('── Embedding Verification ──')for fold in [0, 1, 2]:    npz = OUTPUT_ROOT / 'embeddings' / f'cnn_metseg_embeddings_fold{fold}_v2.npz'    if not npz.exists(): print(f'  ❌ fold{fold}: NOT FOUND'); continue    data = np.load(npz); keys = list(data.keys())    vals = np.stack([data[k] for k in keys])    norms = np.linalg.norm(vals, axis=1)    dim = vals.shape[1]    # Diversity    vn = vals / (norms[:, None] + 1e-8)    n_pairs = min(50, len(keys) * (len(keys)-1)//2)    pairs = random.sample([(i,j) for i in range(len(keys)) for j in range(i+1, len(keys))], n_pairs)    sims = [float(np.dot(vn[i], vn[j])) for i,j in pairs]    cos_mean = np.mean(sims)    diverse = 100 * np.mean(np.array(sims) < 0.95)    status = '✅' if diverse > 20 else '⚠️ low diversity'    print(f'  fold{fold}: {len(keys)} × {dim}-dim | norm [{norms.min():.2f},{norms.max():.2f}] | cos={cos_mean:.3f} diverse={diverse:.0f}% {status}')